In [45]:
# ==========================================
# CELL 1: Setup and Functions
# ==========================================
nvars = 3 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    m1 = z*x1**5
    m2 = z**2*x1**5*x2**5
    m3 = z*x1**4*x2 
    m4 = -z**2*x1**5*x2**5
    m5 = z*x1**3*x2**2
    m6 = z*x1**3*x2*x3
    m7 = -z*x1**3*x2*x3
    m8 = z**2*x1**4*x2**4*x3**2
    m9 = z*x1**2*x2**2*x3
    m10 = -z**2*x1**4*x2**3*x3**3
    m11 = z**2*x1**4*x2**3*x3**3
  
    m12 = -z**3*x1**5*x2**5*x3**5
    m13 = z**3*x1**5*x2**5*x3**5
    m14 = 0#z**2*x1**3*x2**3*x3**3*x4
    m15 = 0#z*x1*x2*x3*x4*x5
    return BR((1-m1)*(1-m2)**2*(1-m3)*(1-m4)*(1-m5)*(1-m6)**2*(1-m7)*(1-m8)*(1-m9)*(1-m10)*(1-m11)*(1-m12)**2*(1-m13))

# Global cache to expand the denominator exactly once
EXPANDED_DENOMINATOR = None

def get_den_expanded():
    global EXPANDED_DENOMINATOR
    if EXPANDED_DENOMINATOR is None:
        EXPANDED_DENOMINATOR = den_guess()
    return EXPANDED_DENOMINATOR

@cached_function
def den_coeff(d):
    global BR, z
    # Extremely fast native coefficient extraction (bypassing SR completely)
    return get_den_expanded().coefficient({z: d})

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

Defining x1, x2, x3, z


In [46]:
# ==========================================
# CELL 2: Execution Loop
# ==========================================
out = 0

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, 300):
    CC = calc_num([2,1,1,1], d)
    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(d, len(CC_list), "FRONT:", front, "BACK:", back)
        else:
            print(d, len(CC_list), CC_list)
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

0 1 [(1, 1)]
1 3 [(1, x1^4*x2), (-1, x1^3*x2*x3), (-2, x1^2*x2^2*x3)]
2 10 FRONT: [(1, x1^8*x2^2), (1, x1^7*x2^3), (2, x1^6*x2^4)] BACK: [(2, x1^5*x2^3*x3^2), (2, x1^4*x2^4*x3^2), (-2, x1^4*x2^3*x3^3)]
3 17 FRONT: [(-1, x1^11*x2^4), (1, x1^10*x2^5), (2, x1^8*x2^7)] BACK: [(4, x1^7*x2^4*x3^4), (2, x1^6*x2^5*x3^4), (1, x1^5*x2^5*x3^5)]
4 25 FRONT: [(-2, x1^13*x2^7), (-1, x1^11*x2^9), (1, x1^10*x2^10)] BACK: [(-1, x1^8*x2^7*x3^5), (2, x1^8*x2^6*x3^6), (-3, x1^7*x2^7*x3^6)]
5 37 FRONT: [(1, x1^16*x2^9), (-2, x1^15*x2^10), (-1, x1^14*x2^11)] BACK: [(-3, x1^11*x2^7*x3^7), (3, x1^9*x2^9*x3^7), (-1, x1^9*x2^8*x3^8)]
6 48 FRONT: [(-1, x1^17*x2^13), (-1, x1^19*x2^10*x3), (-1, x1^18*x2^11*x3)] BACK: [(-1, x1^12*x2^9*x3^9), (1, x1^11*x2^10*x3^9), (1, x1^10*x2^10*x3^10)]
7 53 FRONT: [(-1, x1^21*x2^14), (-1, x1^20*x2^14*x3), (-1, x1^18*x2^16*x3)] BACK: [(-2, x1^14*x2^11*x3^10), (-2, x1^13*x2^12*

KeyboardInterrupt: 

In [49]:
factor(out)

(-1) * (x1^58*x2^43*x3^24*z^25 - 2*x1^56*x2^41*x3^23*z^24 - x1^55*x2^42*x3^23*z^24 + x1^54*x2^42*x3^24*z^24 - 2*x1^54*x2^40*x3^21*z^23 + 2*x1^54*x2^39*x3^22*z^23 + 2*x1^53*x2^40*x3^22*z^23 + 3*x1^52*x2^41*x3^22*z^23 + x1^53*x2^39*x3^23*z^23 + 2*x1^51*x2^41*x3^23*z^23 - x1^53*x2^38*x3^24*z^23 + 2*x1^52*x2^39*x3^24*z^23 + x1^51*x2^40*x3^24*z^23 + x1^50*x2^41*x3^24*z^23 + x1^53*x2^38*x3^19*z^22 + 2*x1^52*x2^38*x3^20*z^22 + 4*x1^51*x2^39*x3^20*z^22 - 2*x1^52*x2^37*x3^21*z^22 - x1^49*x2^40*x3^21*z^22 - x1^51*x2^37*x3^22*z^22 - x1^50*x2^38*x3^22*z^22 - 3*x1^49*x2^39*x3^22*z^22 - 2*x1^48*x2^40*x3^22*z^22 + 3*x1^51*x2^36*x3^23*z^22 - x1^50*x2^37*x3^23*z^22 - 2*x1^49*x2^38*x3^23*z^22 - 2*x1^48*x2^39*x3^23*z^22 - x1^47*x2^40*x3^23*z^22 + 2*x1^50*x2^36*x3^24*z^22 + x1^48*x2^38*x3^24*z^22 - x1^47*x2^39*x3^24*z^22 - 3*x1^51*x2^36*x3^18*z^21 + 2*x1^50*x2^37*x3^18*z^21 - x1^50*x2^36*x3^19*z^21 - 3*x1^49*x2^37*x3^19*z^21 - 5*x1^48*x2^38*x3^19*z^21 + 2*x1^50*x2^35*x3^20*z^21 - 4*x1^48*x2^37*x3^20*z^21 

In [48]:
out/den_guess()

(-x1^58*x2^43*x3^24*z^25 + 2*x1^56*x2^41*x3^23*z^24 + x1^55*x2^42*x3^23*z^24 - x1^54*x2^42*x3^24*z^24 + 2*x1^54*x2^40*x3^21*z^23 - 2*x1^54*x2^39*x3^22*z^23 - 2*x1^53*x2^40*x3^22*z^23 - 3*x1^52*x2^41*x3^22*z^23 - x1^53*x2^39*x3^23*z^23 - 2*x1^51*x2^41*x3^23*z^23 + x1^53*x2^38*x3^24*z^23 - 2*x1^52*x2^39*x3^24*z^23 - x1^51*x2^40*x3^24*z^23 - x1^50*x2^41*x3^24*z^23 - x1^53*x2^38*x3^19*z^22 - 2*x1^52*x2^38*x3^20*z^22 - 4*x1^51*x2^39*x3^20*z^22 + 2*x1^52*x2^37*x3^21*z^22 + x1^49*x2^40*x3^21*z^22 + x1^51*x2^37*x3^22*z^22 + x1^50*x2^38*x3^22*z^22 + 3*x1^49*x2^39*x3^22*z^22 + 2*x1^48*x2^40*x3^22*z^22 - 3*x1^51*x2^36*x3^23*z^22 + x1^50*x2^37*x3^23*z^22 + 2*x1^49*x2^38*x3^23*z^22 + 2*x1^48*x2^39*x3^23*z^22 + x1^47*x2^40*x3^23*z^22 - 2*x1^50*x2^36*x3^24*z^22 - x1^48*x2^38*x3^24*z^22 + x1^47*x2^39*x3^24*z^22 + 3*x1^51*x2^36*x3^18*z^21 - 2*x1^50*x2^37*x3^18*z^21 + x1^50*x2^36*x3^19*z^21 + 3*x1^49*x2^37*x3^19*z^21 + 5*x1^48*x2^38*x3^19*z^21 - 2*x1^50*x2^35*x3^20*z^21 + 4*x1^48*x2^37*x3^20*z^21 + x1^4